In [3]:
import torch   
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

with_cuda = torch.cuda.is_available()
if with_cuda:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import os
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import random_split, DataLoader, Dataset
from torch.amp import GradScaler
from torch.amp import autocast
from torch.optim.lr_scheduler import ReduceLROnPlateau




In [ ]:
def create_mask():   
    path1 = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\data_P3_P4\\P3\\spectrograms_padded"  
    path2 = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\data_P3_P4\\recordings"

    masks = {}
    max_len = 90

    for fichier in os.listdir(path2):
        if fichier.endswith(".wav"):
            path = os.path.join(path2, fichier)
            y, sr = librosa.load(path, sr=None)

            # Calcul du spectrogramme
            stft = librosa.stft(y)
            spectrogram = np.abs(stft)
            log_spectrogram = librosa.amplitude_to_db(spectrogram)

            valid_len = log_spectrogram.shape[1] + 10  # colonnes du vrai spectro
            """valid_len = log_spectrogram.shape[1]   # colonnes du vrai spectro"""
            if valid_len == 0:
                print("valid_len = 0")
            # Création du masque binaire
            mask_2d = np.zeros((1025, 90), dtype=np.float32)
            mask_2d[:, :valid_len] = 1  # 1 sur les colonnes valides


            masks[fichier] = mask_2d  
    return masks  
masks = create_mask()



c:\Users\gabri\anaconda3\envs\gab\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1933
  warnings.warn(
c:\Users\gabri\anaconda3\envs\gab\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1399
  warnings.warn(
c:\Users\gabri\anaconda3\envs\gab\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1876
  warnings.warn(
c:\Users\gabri\anaconda3\envs\gab\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1983
  warnings.warn(
c:\Users\gabri\anaconda3\envs\gab\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=2033
  warnings.warn(
c:\Users\gabri\anaconda3\envs\gab\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1963
  warning

In [5]:
print(masks.keys())

dict_keys(['0_george_0.wav', '0_george_1.wav', '0_george_10.wav', '0_george_11.wav', '0_george_12.wav', '0_george_13.wav', '0_george_14.wav', '0_george_15.wav', '0_george_16.wav', '0_george_17.wav', '0_george_18.wav', '0_george_19.wav', '0_george_2.wav', '0_george_20.wav', '0_george_21.wav', '0_george_22.wav', '0_george_23.wav', '0_george_24.wav', '0_george_25.wav', '0_george_26.wav', '0_george_27.wav', '0_george_28.wav', '0_george_29.wav', '0_george_3.wav', '0_george_30.wav', '0_george_31.wav', '0_george_32.wav', '0_george_33.wav', '0_george_34.wav', '0_george_35.wav', '0_george_36.wav', '0_george_37.wav', '0_george_38.wav', '0_george_39.wav', '0_george_4.wav', '0_george_40.wav', '0_george_41.wav', '0_george_42.wav', '0_george_43.wav', '0_george_44.wav', '0_george_45.wav', '0_george_46.wav', '0_george_47.wav', '0_george_48.wav', '0_george_49.wav', '0_george_5.wav', '0_george_6.wav', '0_george_7.wav', '0_george_8.wav', '0_george_9.wav', '0_jackson_0.wav', '0_jackson_1.wav', '0_jackson_

In [6]:
def normalization(x ,minimum, maximum):
    return (x - minimum) / (maximum - minimum + 1e-8)

def denormalization(x, minimum, maximum):
    return x * (maximum - minimum) + minimum

In [ ]:
path_spec = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\data_P3_P4\\P3\\spectrograms_padded"   

In [9]:
# y'a 6 classes
#taille [1025, 90]


In [13]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout = .1, batch_first = True):
        super(PositionalEncoding, self).__init__()
        self.batch_first = batch_first
        self.dropout = nn.Dropout(p = dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
        pe = torch.zeros(int(max_len), d_model)  
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        if not batch_first:
            pe=pe.transpose(0, 1)
        self.register_buffer('pe', pe)
        
        
    def forward(self, x):
        if x.dim() == 3:
            if self.batch_first:
                return self.dropout(x + self.pe[:, :x.size(1), :])
            else:
                raise ValueError("batch_first must be True")
            
        else : 
            raise ValueError("not implemented yet")
            
        
class ModelTransformer(nn.Transformer):
    def __init__(self, ntokens_dec, max_len_src, max_len_tgt, batch_first = True,
                 d_model = 128, nhead = 8, dim_feedforward = 2048, nlayers = 6, dropout = .1):
        super(ModelTransformer, self).__init__(d_model = d_model, nhead = nhead, dim_feedforward = dim_feedforward, 
                                               num_encoder_layers = nlayers, dropout = dropout,
                                               batch_first = batch_first)
        # Note: self.encoder and self.decoder are already defined in the class nn.Transformer
        self.input_proj = nn.Linear(1025, d_model)

        self.embedding_decode = nn.Embedding(ntokens_dec, d_model)
        self.positional_encoding_enc = PositionalEncoding(d_model, max_len_src)
        self.positional_encoding_dec = PositionalEncoding(d_model, max_len_tgt)
        self.linear = nn.Linear(d_model,ntokens_dec)
        


    def forward(self, src, tgt, src_mask = None, tgt_mask = None,
               src_padding_mask = None, tgt_padding_mask = None):
        
        x = self.input_proj(src)
        x = self.positional_encoding_enc(x)
        x = self.encoder(x, mask=src_mask, src_key_padding_mask = src_padding_mask)
        
        
        y = self.input_proj(tgt)
        y = self.positional_encoding_dec(y)
        res = self.decoder(x,y, tgt_mask=tgt_mask, tgt_key_padding_mask = tgt_padding_mask)
        res = self.linear(res)
        
        
        return nn.functional.log_softmax(res, dim=-1)    
        

In [14]:
def find_min_max(path_spec):
    min_val, max_val = float('inf'), float('-inf')
    count = 0

    for f in os.listdir(path_spec):
        if f.endswith('.npy'):
            data = np.load(os.path.join(path_spec, f), mmap_mode='r')  # Lecture sans tout charger en RAM
            count += 1
            min_val = min(min_val, np.min(data))
            max_val = max(max_val, np.max(data))

    return min_val, max_val, count

"""min_value, max_value, num = find_min_max(path_spec)
print(f"min_value : {min_value}")
print(f"max_value : {max_value}")
print(f"num : {num}")"""

min_value = -77.18305206298828
max_value = 46.953147888183594

In [15]:
def list_spec_files(spec_path):
    return [os.path.join(spec_path, f) for f in os.listdir(spec_path) if f.endswith('.npy')]

def load_spec_normalized(file_path, minim, maxim):
    spec = np.load(file_path)
    spec_tensor = torch.from_numpy(spec).float()
    spec_norm = (spec_tensor - minim) / (maxim - minim)
    return spec_norm

def extract_speaker_name(file_path):
    filename = os.path.basename(file_path)
    return filename.split('_')[0] ,filename.split('_')[1]  # pour "0_nom_43.npy", donne "nom"

def load_dataset(spec_path, minim, maxim, masks):
    files = list_spec_files(spec_path)
    data = []
    for f in files:
        filename = os.path.basename(f).replace('.npy', '.wav')
        spec = load_spec_normalized(f, minim, maxim)
        nombre, speaker = extract_speaker_name(f)
        data.append((spec, masks[filename], nombre ,speaker))
    return data


def get_data(spec_path, minim, maxim, masks, split_ratio=0.8, label = None):
    dataset =load_dataset(spec_path, minim, maxim,masks)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    train_dataset = DataLoader(train_dataset, batch_size=12, shuffle=True)
    test_dataset = DataLoader(test_dataset, batch_size=12, shuffle=False)
    return train_dataset, test_dataset

train_loader, test_loader = get_data(path_spec, min_value, max_value, masks)



In [35]:
def build_loss_vae_log(lambda_reconstruct=0.5, lambda_kl=0.5):
    def loss_vae(x, x_hat, mean, logvar, masks=None):
        
        x_log = torch.log1p(x)
        x_hat_log = torch.log1p(x_hat)
        """x_log = x
        x_hat_log = x_hat"""

        diff_squared = (x_log - x_hat_log) ** 2
        if masks is not None:
            reconstruction_loss = diff_squared[masks.bool()].mean()
        else:
            reconstruction_loss = diff_squared.mean()
        
        logvar2 = torch.clamp(logvar, min=-10, max=10)
    
        kl_loss = -0.5 * torch.sum(1 + logvar2 - mean.pow(2) - logvar2.exp()) / x.shape[0]

        return lambda_reconstruct * reconstruction_loss + lambda_kl * kl_loss

    return loss_vae

def loss_fn(input, target):
    input  = torch.log1p(input)
    target = torch.log1p(target)
    return F.mse_loss(input, target)

def build_loss_vae(lambda_reconstruct=0.5, lambda_kl=0.5):
    def loss_vae(x, x_hat, mean, logvar, masks=None):
        
        x_log = x
        x_hat_log = x_hat

        diff_squared = (x_log - x_hat_log) ** 2
        if masks is not None:
            reconstruction_loss = diff_squared[masks.bool()].mean()
        else:
            reconstruction_loss = diff_squared.mean()
        
        logvar2 = torch.clamp(logvar, min=-10, max=10)
    
        kl_loss = -0.5 * torch.sum(1 + logvar2 - mean.pow(2) - logvar2.exp()) / x.shape[0]

        return lambda_reconstruct * reconstruction_loss + lambda_kl * kl_loss

    return loss_vae

In [ ]:
def create_mask():   
    path1 = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\data_P3_P4\\P3\\spectrograms_padded"  
    path2 = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\data_P3_P4\\recordings"

    masks = {}
    max_len = 90

    for fichier in os.listdir(path2):
        if fichier.endswith(".wav"):
            path = os.path.join(path2, fichier)
            y, sr = librosa.load(path, sr=None)

            # Calcul du spectrogramme
            stft = librosa.stft(y)
            spectrogram = np.abs(stft)
            log_spectrogram = librosa.amplitude_to_db(spectrogram)

            valid_len = log_spectrogram.shape[1]  # colonnes du vrai spectro

            # Création du masque binaire
            mask_2d = np.zeros((1025, 90), dtype=np.float32)
            mask_2d[:, :valid_len] = 1  # 1 sur les colonnes valides


            masks[fichier] = mask_2d  
            return masks  
masks = create_mask()


In [18]:
name_to_index = {'lucas':1, 'theo':2, 'george':3, 'jackson':4, 'nicolas':5, 'yweweler':0}  # 6 classes

index_to_name = {v: k for k, v in name_to_index.items()}

personne = ['lucas', 'theo', 'george', 'jackson', 'nicolas', 'yweweler']
indices = torch.tensor([name_to_index[n] for n in personne], device=device)
y_onehot = torch.nn.functional.one_hot(indices, num_classes=6).float()
idx = torch.argmax(y_onehot[1]).item()
print("Le vecteur one-hot est :", y_onehot[1])
print("Ce vecteur one-hot correspond à :", index_to_name[idx])


Le vecteur one-hot est : tensor([0., 0., 1., 0., 0., 0.], device='cuda:0')
Ce vecteur one-hot correspond à : theo


In [19]:
def avoironehot(personne):
    if len(personne) == 1:
        indices = torch.tensor([name_to_index[personne[0]]], device=device)
        y_onehot = torch.nn.functional.one_hot(indices, num_classes=6).float()
        return y_onehot
    else : 
        indices = torch.tensor([name_to_index[n] for n in personne], device=device)
        y_onehot = torch.nn.functional.one_hot(indices, num_classes=6).float()
        return y_onehot

In [20]:
def save_audio_original_e(path, spec, epoch):
    try:
        # Enregistrer le fichier audio
        audio_path = os.path.join(path + "\\son", f"audio_original.wav")
        spec_amp = librosa.db_to_amplitude(spec)
        signal = librosa.griffinlim(spec_amp, hop_length=512, n_fft=2048)
        sf.write(audio_path, signal, 22050)
    except Exception as e:
        pass
         
def save_audio_reconstructed_e(path, spec, epoch):
    try:
        # Enregistrer le fichier audio
        audio_path = os.path.join(path + "\\son", f"audio_reconstruit{epoch}.wav")
        spec_amp = librosa.db_to_amplitude(spec)
        signal = librosa.griffinlim(spec_amp, hop_length=512, n_fft=2048)
        sf.write(audio_path, signal, 22050)
    except Exception as e:
        pass

def save_image_original_e(path, image, epoch):
    try:
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(image,sr=22050,hop_length=512, n_fft=2048,
            y_axis='log',x_axis='time',vmin=-80, vmax=60,cmap='magma'  
        )
        plt.colorbar(format='%+2.0f dB')
        plt.title(f"Original : {epoch}")
        img_path = os.path.join(path + "\\images", "images_original.png")
        plt.savefig(img_path)
        plt.close()
    except Exception as e:
        pass

def save_image_reconstruit_e(path, image, epoch):
    try:
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(image,sr=22050,hop_length=512, n_fft=2048,
            y_axis='log',x_axis='time',vmin=-80, vmax=60,cmap='magma'  
        )
        plt.colorbar(format='%+2.0f dB')
        plt.title(f"Reconstruit : {epoch}")
        img_path = os.path.join(path + "\\images", f"images_reconstuites{epoch}.png")
        plt.savefig(img_path)
        plt.close()
    except Exception as e:
        pass


In [ ]:
def testeur(data_loader, model, device , epoch , loss_test, loss_affichage, plot_meanmin=None, plot_logvarmin = None, plot_logvarmax = None, plot_meanmax = None, minim = None, maxim = None):
    path = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P3\\test"
    
    
    original, masks, nombre, personne = data_loader.dataset[12]
    original = original.unsqueeze(0).to(device)
   
    with torch.no_grad():
        personne = [personne]
        y_onehot = avoironehot(personne)
        y_onehot = y_onehot.to(device)
        reconstructed, mean, logvar = model(original, y_onehot) # 120 200 1025

    original = original.squeeze(0).cpu().detach().numpy()
    
    reconstructed = reconstructed.squeeze(0).cpu().detach().numpy()
    reconstructed = reconstructed.reshape(1025, 90)

    original = denormalization(original, minim, maxim)

    reconstructed2 = denormalization(reconstructed, minim, maxim)
    original = original * masks
    
    col_valides = masks[0] > 0  # 0ᵉ ligne suffit car tout est répété verticalement

    # Supprimer les colonnes paddées (axe 1)
    original = original[:, col_valides]
    reconstructed2 = reconstructed2[:, col_valides]
    
    save_audio_original_e(path, original, epoch)
    save_audio_reconstructed_e(path, reconstructed2, epoch)
    save_image_original_e(path, original, epoch)
    save_image_reconstruit_e(path, reconstructed2, epoch)
    
    if epoch < 11 : 
        plt.figure(figsize=(10, 4))
        train = list(range(len(loss_affichage)))         # [0, 1, 2, ..., 49]
        test = list(range(0, epoch+1 , 10))   # [0, 10, 20, 30, 40]
   
        # 2. Les courbes
        plt.plot(train, loss_affichage, label='Train Loss', color='blue')
        plt.plot(test, loss_test, label='Test Loss', color='orange', marker='o')

        # 3. Habillage
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.title('Train vs Test Loss')
        plt.legend()
        plt.grid(True)

        # 4. Sauvegarde
        plt.savefig(os.path.join(path + "\\loss", f"loss_comparaison_{epoch}epochs.png"))
        plt.close()      
    
    
    else: 
        loss__test_tronque = loss_test[1:]
        loss_affichage_tronque = loss_affichage[10:]
        plt.figure(figsize=(10, 4))
        train = list(range(len(loss_affichage_tronque)))         # [0, 1, 2, ..., 49]
        test = list(range(0, epoch, 10))   # [0, 10, 20, 30, 40]

        # 2. Les courbes
        plt.plot(train, loss_affichage_tronque, label='Train Loss', color='blue')
        plt.plot(test, loss__test_tronque, label='Test Loss', color='orange', marker='o')

        # 3. Habillage
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.title('Train vs Test Loss')
        plt.legend()
        plt.grid(True)

        # 4. Sauvegarde
        plt.savefig(os.path.join(path + "\\loss", f"loss_comparaison_tronque{epoch}.png"))
        plt.close()
        
# Figure pour logvar min/max
        
        plt.subplot(1, 2, 1)
        plt.plot(plot_logvarmin[200:], label='LogVar Min', color='orange')
        plt.title('Log Variance Min')
        plt.xlabel('Epochs')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True)

        plt.subplot(1, 2, 2)
        plt.plot(plot_logvarmax[200:], label='LogVar Max', color='red')
        plt.title('Log Variance Max')
        plt.xlabel('Epochs')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.savefig(os.path.join(path, "loss", f"logvar_minmax.png"))
        plt.close()

        # Figure pour mean min/max
        plt.figure(figsize=(200, 4))
        plt.subplot(1, 2, 1)
        plt.plot(plot_meanmin[10:], label='Mean Min', color='blue')
        plt.title('Mean Min')
        plt.xlabel('Epochs')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True)

        plt.subplot(1, 2, 2)
        plt.plot(plot_meanmax[200:], label='Mean Max', color='green')
        plt.title('Mean Max')
        plt.xlabel('Epochs')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.savefig(os.path.join(path, "loss", f"mean_minmax.png"))
        plt.close()

     
    

In [23]:
def cond_train_model_transfo(data_loader, model, criterion, optimizer, nepochs, scheduler, num_classes=6):
    #List to store loss to visualize
    train_losses = []
    train_acc = []
    start_epoch = 0
    loss_test = []
    plot_meanmin = []
    plot_logvarmin = []
    plot_meanmax = []
    plot_logvarmax = []
    for epoch in range(start_epoch, nepochs):
        train_loss = 0.
        valid_loss = 0.
        correct = 0

        model.train()
        for batch_idx, (input_, mask, nombre, personne) in enumerate(tqdm(data_loader, desc="Batch", leave=False)):
            input_ = input_.to(device)
            
            # clear the gradients of all optimized variables
            optimizer.zero_grad()
            mask = mask.to(device)
            # convert batch of 'personne' to one-hot tensor
            y_onehot = avoironehot(personne)
            y_onehot = y_onehot.to(device)
            # forward pass: compute predicted outputs by passing inputs to the model
            # Autoencodeur simple
            
            
            # 1) Masque de padding temporel (B, T)
            #    True là où la colonne temporelle t est entièrement paddée
            pad_mask_time = (mask.sum(dim=1) == 0)    # shape (B, T)

            # 2) src_key_padding_mask : (B, T)
            src_key_padding_mask = pad_mask_time

            # 3) tgt_key_padding_mask : (B, T-1) si tu shifts d’un pas
            tgt_key_padding_mask = pad_mask_time

            # 4) tgt_mask (causal) : interdit j>i, shape (T-1, T-1)
            T_tgt = src_key_padding_mask.size(1) 
            tgt_mask = torch.triu(
                torch.ones(T_tgt, T_tgt, device=device),
                diagonal=1
            ).bool()
            
            # src_padding_mask = (mask_src == 0)   
            # tgt_padding_mask ?
            # ajout du changement de personne ?
            
            input_ = input_.permute(0, 2, 1) # [B, 90, 1025]
            
            output = model(
                src=input_, 
                tgt=input_,
                src_padding_mask=src_key_padding_mask,
                tgt_padding_mask=tgt_key_padding_mask,

            )
            
            mask = mask.permute(0, 2, 1) # changer mask 
            
            loss = ((output - input_)**2 * mask).mean()

            loss.backward()
            # Dans la boucle d'entraînement :

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Évite les gradients explosifs

            optimizer.step()
            
            

            # update training loss
            train_loss += loss.item() * input_.size(0)

        # calculate average losses
        train_loss = train_loss/len(data_loader.dataset)
        train_losses.append(train_loss)
        if scheduler is not None:
            scheduler.step(train_loss)
        # print losses statistics 
        print('Epoch: {} \tTraining Loss: {:.6f}'.format(
            epoch, train_loss))
        if epoch % 10 == 0:
            test_loss = 0
            model.eval()
            
            with torch.no_grad():
                for batch_idx, (input_, mask, nombre, personne) in enumerate(test_loader):
                    input_ = input_.to(device)
                    y_onehot = avoironehot(personne)
                    y_onehot = y_onehot.to(device)
                    mask = mask.to(device)
                    # forward pass: compute predicted outputs by passing inputs to the model
                    output, mean, logvar = model(input_, y_onehot)
                    """input_ = input_.view(input_.size(0), -1)"""
                    output = output.squeeze(1)
                    loss = criterion(input_, output, mean, logvar, mask)
                    test_loss += loss.detach().cpu().item()
                loss_test.append(test_loss/len(test_loader))
                print(f"Test loss = {test_loss/len(test_loader):.6f}")
            testeur(train_loader, model, device, epoch, loss_test, train_losses, plot_logvarmin, plot_meanmin, plot_logvarmax, plot_meanmax ,min_value, max_value)





In [36]:
# model
input_dim = 1025 * 90  # = 92250
cond_dim = 6  # nombre de classes


ntokens_dec = 1025   # plus de ntockens_enc ajout de input_proj
max_len_src = 90     
max_len_tgt = 90     


                 
model = ModelTransformer(
    ntokens_dec=ntokens_dec,
    max_len_src=max_len_src,
    max_len_tgt=max_len_tgt
    ).to(device)

# loss function
criterion = build_loss_vae()

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)


nepochs = 51

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

cond_train_model_transfo(train_loader,model, criterion, optimizer, nepochs, scheduler= scheduler)

PATH = "model2.pth"

# Sauvegarder les paramètres du modèle
torch.save(model.state_dict(), PATH)


c:\Users\gabri\anaconda3\envs\gab\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch: 0 	Training Loss: 2.464355
Test loss = 0.005266


Epoch: 1 	Training Loss: 0.004628


Epoch: 2 	Training Loss: 0.004418


Epoch: 3 	Training Loss: 0.004379


...

Epoch: 28 	Training Loss: 0.004194


Epoch: 29 	Training Loss: 0.004189


Epoch: 30 	Training Loss: 0.004185
Test loss = 0.004322


Epoch: 31 	Training Loss: 0.004184


KeyboardInterrupt: 